# data-cleaning_005

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (115).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `forex` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 2 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/Forex Data/EURGBP/DAT_MT_EURGBP_M1_2020.csv")
data

In [ ]:
import pandas as pd
import os
from glob import glob

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Set path to Forex data folder
data_path = '/content/drive/MyDrive/Forex Data/EURGBP'

# Step 3: Get all file paths
file_list = glob(os.path.join(data_path, '*'))

# Step 4: Read and clean each file
df_list = []

for file in file_list:
    try:
        # Read the first line to detect delimiter
        with open(file, 'r', encoding='utf-8', errors='ignore') as f:
            first_line = f.readline()

        # Try to guess delimiter
        if '\t' in first_line:
            delimiter = '\t'
        elif ',' in first_line:
            delimiter = ','
        elif ';' in first_line:
            delimiter = ';'
        else:
            delimiter = r'\s+'

        # Load the file
        df = pd.read_csv(file, sep=delimiter, header=None, engine='python')

        # If the file loaded with only 1 column, try to split it
        if df.shape[1] == 1:
            df = df[0].str.split(delimiter, expand=True)

        # Expecting 7 columns: Date, Time, Open, High, Low, Close, Volume
        if df.shape[1] != 7:
            print(f"⚠️ Skipping {file} due to unexpected format ({df.shape[1]} columns)")
            continue

        # Assign column names
        df.columns = ['Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume']

        df_list.append(df)

    except Exception as e:
        print(f"❌ Error reading {file}: {e}")

# Step 5: Combine all dataframes
combined_df = pd.concat(df_list, ignore_index=True)

# Step 6: Create datetime column and sort
combined_df['Datetime'] = pd.to_datetime(combined_df['Date'] + ' ' + combined_df['Time'], errors='coerce')
combined_df.dropna(subset=['Datetime'], inplace=True)
combined_df.sort_values('Datetime', inplace=True)
combined_df.reset_index(drop=True, inplace=True)

combined_df['Ticker'] = 'EURGBP'

# Step 7: Reorder columns
combined_df = combined_df[['Ticker', 'Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']]

# Step 8: Show result
print("✅ Combined shape:", combined_df.shape)
display(combined_df.head())

combined_df.to_csv('/content/drive/MyDrive/Forex Data/EURGBP/EURGBP_min_2020_202505.csv', index=False)
print("📁 Saved as combined_forex_data.csv")
